# Notebook 02 – Preprocessing & Advanced Feature Engineering

**Purpose:** Demonstrate and validate every preprocessing and feature engineering
step before it enters any model. All transforms are wrapped in sklearn Pipelines
to guarantee **no data leakage**.

**Split:** Stratified 60 / 20 / 20 (train / validation / test), `random_state=42`.

In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

from data_utils import load_raw, make_splits, class_balance
from features   import (
    ChurnFeatureEngineer, build_preprocessor,
    build_linear_pipeline, build_tree_pipeline,
    NUMERIC_COLS, CATEGORICAL_COLS, CONTRACT_COL
)

FIG_DIR    = Path('../reports/figures')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(exist_ok=True)

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams['savefig.dpi'] = 300

def savefig(name):
    plt.savefig(FIG_DIR / name, dpi=300, bbox_inches='tight')
    plt.show()

## 1. Stratified 60 / 20 / 20 Split

In [ ]:
df = load_raw()
X_train, X_val, X_test, y_train, y_val, y_test = make_splits(df)

print(f'Train : {X_train.shape}  churn={y_train.mean():.3f}')
print(f'Val   : {X_val.shape}    churn={y_val.mean():.3f}')
print(f'Test  : {X_test.shape}   churn={y_test.mean():.3f}')
print('\nStratification check: churn rates should be ~0.265 in all splits')

**Insight:** Stratified splitting preserves the ~26.5% churn rate in every split,
ensuring that CV scores are not inflated or deflated by lucky class distributions.

## 2. Apply Feature Engineering (ChurnFeatureEngineer)

In [ ]:
fe = ChurnFeatureEngineer()
# fit() is a no-op (stateless), but we call it for Pipeline compatibility
X_train_fe = fe.fit_transform(X_train)
X_val_fe   = fe.transform(X_val)
X_test_fe  = fe.transform(X_test)

new_cols = [
    'tenure_group', 'num_services', 'has_protection_bundle',
    'has_streaming', 'avg_monthly_spend', 'charge_gap',
    'is_month_to_month', 'is_autopay', 'contract_internet',
    'monthly_charges_per_service', 'is_new_customer', 'is_family'
]
print(f'Columns after FE: {X_train_fe.shape[1]}  (was {X_train.shape[1]})')
X_train_fe[new_cols].head()

## 3. Validate Each Engineered Feature

In [ ]:
plot_df = X_train_fe[new_cols].copy()
plot_df['Churn'] = y_train.values

# Churn rate for each binary/count feature
binary_count_cols = [
    'num_services', 'has_protection_bundle', 'has_streaming',
    'is_month_to_month', 'is_autopay', 'is_new_customer', 'is_family'
]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(binary_count_cols):
    rates = plot_df.groupby(col)['Churn'].mean()
    counts = plot_df.groupby(col)['Churn'].count()
    bars = axes[i].bar(rates.index.astype(str), rates.values,
                       color='#D65F5F', alpha=0.85, edgecolor='white')
    for bar, idx in zip(bars, rates.index):
        axes[i].text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + 0.01,
                     f'n={counts[idx]}', ha='center', fontsize=7)
    axes[i].axhline(y_train.mean(), color='navy', linestyle='--', linewidth=1)
    axes[i].set_title(col, fontsize=9, fontweight='bold')
    axes[i].set_ylabel('Churn Rate')

axes[-1].set_visible(False)
plt.suptitle('Engineered Features vs Churn Rate (Train Set)', fontsize=12, fontweight='bold')
plt.tight_layout()
savefig('08_engineered_features_churn.png')

In [ ]:
# Continuous engineered features: KDE by churn
cont_cols = ['avg_monthly_spend', 'charge_gap', 'monthly_charges_per_service']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, cont_cols):
    for churn_val, label, color in [(0,'No Churn','#4878CF'), (1,'Churn','#D65F5F')]:
        subset = plot_df[plot_df['Churn'] == churn_val][col]
        subset.plot.kde(ax=ax, label=label, color=color, linewidth=2)
    ax.set_title(col, fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('Continuous Engineered Features – KDE by Churn', fontsize=12, fontweight='bold')
plt.tight_layout()
savefig('09_engineered_continuous_kde.png')

In [ ]:
# Tenure group churn rates
fig, ax = plt.subplots(figsize=(8, 4))
rates = plot_df.groupby('tenure_group')['Churn'].mean()
counts = plot_df.groupby('tenure_group')['Churn'].count()
bars = ax.bar(rates.index.astype(str), rates.values,
              color='#D65F5F', alpha=0.85, edgecolor='white')
for bar, idx in zip(bars, rates.index):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.01,
            f'n={counts[idx]}', ha='center', fontsize=8)
ax.axhline(y_train.mean(), color='navy', linestyle='--', label='Overall avg')
ax.set_xlabel('Tenure Group (months)')
ax.set_ylabel('Churn Rate')
ax.set_title('Churn Rate by Tenure Group', fontweight='bold')
ax.legend()
plt.tight_layout()
savefig('10_tenure_group_churn.png')

In [ ]:
# contract_internet interaction: top 10 combinations by churn rate
interaction_rates = (
    plot_df.groupby('contract_internet')['Churn']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'churn_rate', 'count': 'n'})
    .query('n >= 20')
    .sort_values('churn_rate', ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(interaction_rates.index, interaction_rates['churn_rate'],
               color='#D65F5F', alpha=0.85, edgecolor='white')
ax.axvline(y_train.mean(), color='navy', linestyle='--', label='Overall avg')
ax.set_xlabel('Churn Rate')
ax.set_title('Churn Rate by Contract × Internet Service Interaction', fontweight='bold')
ax.legend()
plt.tight_layout()
savefig('11_contract_internet_interaction.png')
print(interaction_rates.to_string())

**Insight:** Month-to-month + Fiber optic customers churn at the highest rate
(~55%), confirming the interaction feature captures a real joint effect that
neither feature alone fully represents.

## 4. Preprocessing Pipeline (ColumnTransformer)

In [ ]:
# Build and fit the preprocessor on TRAINING DATA ONLY
# (validation and test sets are only transformed, never used to fit)
prep_linear = build_preprocessor(scale=True)   # for LR / SVM
prep_tree   = build_preprocessor(scale=False)  # for RF / XGB / LGBM

X_train_linear = prep_linear.fit_transform(X_train_fe)
X_val_linear   = prep_linear.transform(X_val_fe)
X_test_linear  = prep_linear.transform(X_test_fe)

X_train_tree   = prep_tree.fit_transform(X_train_fe)
X_val_tree     = prep_tree.transform(X_val_fe)
X_test_tree    = prep_tree.transform(X_test_fe)

print(f'Linear preprocessed shape: {X_train_linear.shape}')
print(f'Tree   preprocessed shape: {X_train_tree.shape}')

In [ ]:
# Inspect feature names after OHE
feature_names = prep_linear.get_feature_names_out()
print(f'Total features after preprocessing: {len(feature_names)}')
print('\nFirst 20 feature names:')
print(feature_names[:20])

## 5. Imbalance Handling: class_weight vs SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Strategy A: class_weight='balanced' (equivalent to cost-sensitive learning)
pipe_cw = build_linear_pipeline(
    LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
)
scores_cw = cross_val_score(pipe_cw, X_train, y_train,
                             cv=cv, scoring='average_precision', n_jobs=-1)

# Strategy B: SMOTE inside CV folds (via imblearn Pipeline)
# CRITICAL: SMOTE must be INSIDE the pipeline so it only sees training folds,
# never the validation fold. This prevents synthetic samples from leaking.
smote_pipe = ImbPipeline([
    ('fe',    ChurnFeatureEngineer()),
    ('prep',  build_preprocessor(scale=True)),
    ('smote', SMOTE(random_state=42)),
    ('model', LogisticRegression(max_iter=1000, random_state=42)),
])
scores_smote = cross_val_score(smote_pipe, X_train, y_train,
                                cv=cv, scoring='average_precision', n_jobs=-1)

print(f'class_weight=balanced  PR-AUC: {scores_cw.mean():.4f} ± {scores_cw.std():.4f}')
print(f'SMOTE                  PR-AUC: {scores_smote.mean():.4f} ± {scores_smote.std():.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
strategies = ['class_weight=balanced', 'SMOTE (inside CV)']
means  = [scores_cw.mean(), scores_smote.mean()]
stds   = [scores_cw.std(),  scores_smote.std()]
bars = ax.bar(strategies, means, yerr=stds, capsize=6,
              color=['#4878CF', '#D65F5F'], alpha=0.85, edgecolor='white')
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.005,
            f'{m:.4f}', ha='center', fontweight='bold')
ax.set_ylabel('PR-AUC (5-fold CV)')
ax.set_title('Imbalance Strategy Comparison (Logistic Regression)', fontweight='bold')
ax.set_ylim(0, min(1.0, max(means) + 0.1))
plt.tight_layout()
savefig('12_imbalance_strategy_comparison.png')

**Decision:** We use `class_weight='balanced'` (or `scale_pos_weight` for XGBoost)
as the primary imbalance strategy. It is simpler, faster, and avoids the risk of
SMOTE introducing synthetic artefacts in a small dataset. SMOTE is kept as a
comparison baseline in the modeling notebook.

## 6. Save Fitted Preprocessing Pipelines

In [ ]:
# Save the fitted preprocessors so they can be loaded in later notebooks
# without re-fitting (which would cause leakage if done on val/test data)
joblib.dump(prep_linear, MODELS_DIR / 'preprocessor_linear.joblib')
joblib.dump(prep_tree,   MODELS_DIR / 'preprocessor_tree.joblib')

# Also save the split indices for reproducibility
split_info = {
    'train_idx': X_train.index.tolist(),
    'val_idx':   X_val.index.tolist(),
    'test_idx':  X_test.index.tolist(),
}
joblib.dump(split_info, MODELS_DIR / 'split_indices.joblib')

print('Saved:')
print('  models/preprocessor_linear.joblib')
print('  models/preprocessor_tree.joblib')
print('  models/split_indices.joblib')

## 7. Preprocessing Summary Table

In [ ]:
summary = pd.DataFrame([
    {'Step': 'TotalCharges fix',       'Detail': 'pd.to_numeric(errors=coerce) → fillna(0)',                    'Why': 'New customers (tenure=0) have no billing history'},
    {'Step': 'Drop customerID',        'Detail': 'df.drop(columns=["customerID"])',                             'Why': 'Unique ID, zero predictive signal, leakage risk'},
    {'Step': 'Normalise categoricals', 'Detail': '"No internet service" / "No phone service" → "No"',           'Why': 'Redundant encoding; reduces OHE cardinality'},
    {'Step': 'Stratified split',       'Detail': '60/20/20, stratify=y, random_state=42',                       'Why': 'Preserves 26.5% churn rate in all splits'},
    {'Step': 'Feature engineering',    'Detail': '12 new features (see features.py)',                           'Why': 'Capture non-linear and interaction effects'},
    {'Step': 'Numeric imputation',     'Detail': 'SimpleImputer(strategy=median)',                              'Why': 'Robust to outliers; no missing values in this dataset but defensive'},
    {'Step': 'Scaling (linear only)',  'Detail': 'StandardScaler on numeric cols',                              'Why': 'LR/SVM are distance-based; trees do not need scaling'},
    {'Step': 'OHE (categorical)',      'Detail': 'OneHotEncoder(handle_unknown=ignore, drop=if_binary)',        'Why': 'Handles unseen categories at inference; avoids dummy trap'},
    {'Step': 'Contract encoding',      'Detail': 'OrdinalEncoder: Two year=0, One year=1, Month-to-month=2',   'Why': 'Natural ordinal churn-risk ordering'},
    {'Step': 'Imbalance handling',     'Detail': 'class_weight=balanced (primary); SMOTE inside CV (comparison)', 'Why': 'Avoids accuracy paradox; SMOTE kept inside folds to prevent leakage'},
])
summary

In [ ]:
summary.to_csv('../reports/results/preprocessing_summary.csv', index=False)
print('Saved preprocessing_summary.csv')
print('\nNotebook 02 complete.')